In [10]:
# ============================================================
# TRANSFORM RESULTS DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [11]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [12]:
# -----------------------------------------
# 2. Receive p_batch_id (unified logic)
# -----------------------------------------

# Case 1: Papermill parameter
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# Case 3: Manual Jupyter execution → auto-detect latest batch
else:
    bronze_folder = f"{BRONZE_PATH}/results/data.parquet"
    batch_folders = [
        f.split("=")[1]
        for f in os.listdir(bronze_folder)
        if f.startswith("batch_id=")
    ]
    if not batch_folders:
        raise Exception("❌ No batch_id partitions found in Bronze results")
    p_batch_id = sorted(batch_folders)[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)

Using p_batch_id: 20260609_132033


In [13]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/results/data.parquet"
silver_path = f"{SILVER_PATH}/results"
os.makedirs(silver_path, exist_ok=True)

In [14]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
results_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze results read")
results_df.show(5, truncate=False)

✔ Bronze results read
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+--------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|date      |raceName            |round|season|url                                                    |constructorId|driverId|grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                   |batch_id       |
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+--------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|1989-03-26|brazilian grand prix|1    |1989  |https://en.wikipedia.org/wiki/1989_Brazilian_Grand_Prix|ferrari      |m

In [15]:
# -----------------------------------------
# 5. Select required columns + standardize names
# -----------------------------------------
results_selected_df = (
    results_df
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName",
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingest_timestamp",
            "source_file",
            "batch_id"
        )
        .withColumnsRenamed(
            {
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "final_position",
                "positionText": "final_position_text"
            }
        )
)

In [16]:
# -----------------------------------------
# 6. Business key validation + remove duplicates
# -----------------------------------------
results_valid_df = (
    results_selected_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [17]:
# -----------------------------------------
# 7. Title Case transformation
# -----------------------------------------
results_final_df = (
    results_valid_df
        .withColumn("race_name", F.initcap("race_name"))
)

In [21]:
# -----------------------------------------
# 9. Validate
# -----------------------------------------
if not os.path.exists(f"{silver_path}/data.parquet"):
    print("⚠️ Silver Results not written — check batch_id or input data")
else:
    spark.read.parquet(f"{silver_path}/data.parquet").show(5, truncate=False)


⚠️ Silver Results not written — check batch_id or input data
